# RMSNorm 手撕实现

## 定义
$$\text{RMSNorm}(x)=\frac{x}{\sqrt{\frac1d\sum_i x_i^2+\epsilon}}\odot \gamma$$
- 相比 LayerNorm **省去减均值**，只按均方根缩放，计算量少约 $7\%\sim64\%$。
- 论文证明 RMSNorm 在效果上与 LayerNorm 相当，LLaMA/Qwen 等默认采用。
- $\gamma$ 是可学习缩放（无平移 $\beta$），$\epsilon$ 防除 0。
- 用 `rsqrt` 一次算出倒数平方根，比 `sqrt` 再除更快。

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # rsqrt = 1/sqrt，一次算倒数；mean(x^2) 即 RMS^2
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight

In [ ]:
# 验证：归一化后 RMS≈1；与 LayerNorm 对比；梯度可反传
torch.manual_seed(0)
dim = 64
rms = RMSNorm(dim)
x = torch.randn(2, 10, dim)
out = rms(x)
rms_val = out.detach().pow(2).mean(-1).sqrt()
print('归一化后 RMS≈1:', rms_val.mean().item(), '(weight=1 时)')

# 与 LayerNorm 对比（量级、是否零均值）
ln = nn.LayerNorm(dim)
print('LayerNorm 零均值:', ln(x).mean(-1).abs().max().item(), '(≈0)')
print('RMSNorm 一般非零均值:', out.detach().mean(-1).abs().max().item())

# 梯度
out.sum().backward()
print('weight.grad 非空:', rms.weight.grad is not None, 'shape', tuple(rms.weight.grad.shape))

## 小结 / 易错点
- 用 `rsqrt` 而非 `1/sqrt`，少一次除法，kernel 友好。
- RMSNorm 不去均值，故输出均值一般非 0，这是与 LayerNorm 的本质区别。
- $\epsilon$ 通常 $10^{-6}$；fp16 训练时有时用 $10^{-5}$ 防下溢。
- 大模型常在 RMSNorm 后再加 QK-Norm（对 q/k 再做一次 RMSNorm）稳训练。

## ✅ 测试验证

In [ ]:
# 验证 RMSNorm
import torch
import torch.nn.functional as F

# RMSNorm: y = x / sqrt(mean(x^2) + eps) * weight
# 关键性质: 不减均值（对比 LayerNorm）

x = torch.randn(2, 8, 32)
eps = 1e-6

# 手动 RMSNorm
rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + eps)
rms_norm = x / rms
print("  ✓ RMSNorm 输出 RMS ≈ 1:", (rms_norm ** 2).mean(dim=-1).mean().item())

# 对比 LayerNorm (减均值)
ln = F.layer_norm(x, (32,))
# RMSNorm 不减均值，所以与 LN 不同（除非 x 均值为 0）
print(f"  ✓ RMSNorm vs LayerNorm 差异: {(rms_norm - ln).abs().mean().item():.6f} (非零因为 RMSNorm 不减均值)")

# 验证: 若 x 均值恰好为 0，RMSNorm ≈ LayerNorm
x_zero_mean = x - x.mean(dim=-1, keepdim=True)
rms_zm = x_zero_mean / torch.sqrt((x_zero_mean ** 2).mean(dim=-1, keepdim=True) + eps)
ln_zm = F.layer_norm(x_zero_mean, (32,))
assert torch.allclose(rms_zm, ln_zm, atol=1e-4), "RMSNorm != LayerNorm when mean=0"
print("  ✓ 均值为0时 RMSNorm == LayerNorm")

print("✅ RMSNorm 测试通过")
